In [ ]:
# ========================================================
# 05_lstm_autoencoder.ipynb
# LSTM Autoencoder po rozszerzeniu normalnego datasetu
# WINDOW_SIZE = 5, STEP = 1
# ========================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# ========================================================
# Ustawienia
# ========================================================

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 5
STEP = 1
BATCH_SIZE = 256
HIDDEN_DIM = 64
LSTM_LAYERS = 1
EPOCHS = 30
LEARNING_RATE = 0.001
THRESHOLD_QUANTILE = 0.95

print("=== LSTM AUTOENCODER v2 - PO ROZSZERZENIU NORMAL DATASET ===")
print("Urządzenie:", DEVICE)
print("WINDOW_SIZE:", WINDOW_SIZE)
print("STEP:", STEP)
print("BATCH_SIZE:", BATCH_SIZE)
print("HIDDEN_DIM:", HIDDEN_DIM)
print("EPOCHS:", EPOCHS)

# ========================================================
# 1. Dataset okien sekwencyjnych
# ========================================================

class WindowDataset(Dataset):
    def __init__(self, csv_path, window_size=WINDOW_SIZE, step=STEP):
        df = pd.read_csv(csv_path)

        self.data = df.values.astype(np.float32)
        self.window_size = window_size
        self.step = step

        self.indices = list(range(0, len(self.data) - window_size + 1, step))

        if len(self.indices) == 0:
            raise ValueError(
                f"Za mało danych w {csv_path}: "
                f"{len(self.data)} flowów, window_size={window_size}"
            )

        print(f"\nPlik: {csv_path}")
        print(f"  flowy: {len(self.data)}")
        print(f"  okna:  {len(self.indices)}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.window_size
        return torch.from_numpy(self.data[start:end])


# ========================================================
# 2. Dane treningowe normal
# ========================================================

normal_path = "../data/processed/normal_features.csv"

normal_df = pd.read_csv(normal_path)
input_dim = normal_df.shape[1]

print("\n=== DANE NORMALNE ===")
print("Liczba cech wejściowych:", input_dim)
print("Liczba flowów normalnych:", len(normal_df))

train_dataset = WindowDataset(
    normal_path,
    window_size=WINDOW_SIZE,
    step=STEP
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

print("Liczba sekwencji treningowych:", len(train_dataset))


# ========================================================
# 3. Model LSTM Autoencoder
# ========================================================

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=HIDDEN_DIM, num_layers=LSTM_LAYERS):
        super().__init__()

        self.encoder = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.decoder = nn.LSTM(
            hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc_out = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        # x: [batch, seq_len, input_dim]
        enc_out, (h, c) = self.encoder(x)

        # ostatni stan ukryty powtarzamy na długość sekwencji
        repeated = h[-1].unsqueeze(1).repeat(1, x.shape[1], 1)

        dec_out, _ = self.decoder(repeated)
        out = self.fc_out(dec_out)

        return out


model = LSTMAutoencoder(
    input_dim=input_dim,
    hidden_dim=HIDDEN_DIM,
    num_layers=LSTM_LAYERS
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

criterion = nn.MSELoss()

print("\n=== MODEL ===")
print(model)


# ========================================================
# 4. Trening
# ========================================================

best_loss = float("inf")
os.makedirs("../models", exist_ok=True)

print("\n=== TRENING LSTM AE ===")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(DEVICE)

        recon = model(batch)
        loss = criterion(recon, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.size(0)

    avg_loss = total_loss / len(train_dataset)

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "../models/lstm_ae_window5_best.pth")

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.6f}")

print("\nTrening zakończony.")
print("Najlepszy loss:", round(best_loss, 6))
print("Najlepszy model zapisany jako: ../models/lstm_ae_window5_best.pth")

model.load_state_dict(torch.load("../models/lstm_ae_window5_best.pth", map_location=DEVICE))
model.eval()


# ========================================================
# 5. Liczenie błędów rekonstrukcji dla okien
# ========================================================

def compute_window_errors(csv_path, window_size=WINDOW_SIZE, step=STEP, batch_size=2048):
    dataset = WindowDataset(csv_path, window_size=window_size, step=step)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    errors_all = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            recon = model(batch)

            # błąd rekonstrukcji dla całego okna:
            # średnia po czasie i cechach
            errors = torch.mean((batch - recon) ** 2, dim=(1, 2))

            errors_all.append(errors.cpu())

    return torch.cat(errors_all).numpy(), len(dataset)


# ========================================================
# 6. Próg na podstawie normal
# ========================================================

normal_errors, normal_windows = compute_window_errors(normal_path)

threshold = np.quantile(normal_errors, THRESHOLD_QUANTILE)

print("\n=== STATYSTYKI LSTM DLA NORMAL ===")
print("Liczba okien normal:", normal_windows)
print("Mean error:", round(float(np.mean(normal_errors)), 6))
print("Median error:", round(float(np.median(normal_errors)), 6))
print("Max error:", round(float(np.max(normal_errors)), 6))
print(f"Threshold q={THRESHOLD_QUANTILE}:", round(float(threshold), 6))


# ========================================================
# 7. Ewaluacja scenariuszy anomalnych
# ========================================================

scenarios = [
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer"
]

results = []

# normal jako FPR
normal_detected = int((normal_errors > threshold).sum())

results.append({
    "scenario": "normal (FPR)",
    "window_size": WINDOW_SIZE,
    "step": STEP,
    "mean_error": float(np.mean(normal_errors)),
    "median_error": float(np.median(normal_errors)),
    "max_error": float(np.max(normal_errors)),
    "LSTM_detected": normal_detected,
    "LSTM_total_windows": normal_windows,
    "LSTM AE DR (%)": round(normal_detected / normal_windows * 100, 2)
})

print("\n=== EWALUACJA LSTM AE ===")
print(
    f"{'normal (FPR)':15s} | detected: "
    f"{normal_detected:8d}/{normal_windows:<8d} "
    f"| DR: {round(normal_detected / normal_windows * 100, 2):6.2f}%"
)

for sc in scenarios:
    path = f"../data/processed/{sc}_features.csv"

    try:
        errors, total_windows = compute_window_errors(path)

        detected = int((errors > threshold).sum())

        dr = round(detected / total_windows * 100, 2)

        results.append({
            "scenario": sc,
            "window_size": WINDOW_SIZE,
            "step": STEP,
            "mean_error": float(np.mean(errors)),
            "median_error": float(np.median(errors)),
            "max_error": float(np.max(errors)),
            "LSTM_detected": detected,
            "LSTM_total_windows": total_windows,
            "LSTM AE DR (%)": dr
        })

        print(
            f"{sc:15s} | detected: "
            f"{detected:8d}/{total_windows:<8d} "
            f"| DR: {dr:6.2f}%"
        )

    except ValueError as e:
        print(f"{sc:15s} | POMINIĘTO: {e}")

        results.append({
            "scenario": sc,
            "window_size": WINDOW_SIZE,
            "step": STEP,
            "mean_error": np.nan,
            "median_error": np.nan,
            "max_error": np.nan,
            "LSTM_detected": 0,
            "LSTM_total_windows": 0,
            "LSTM AE DR (%)": np.nan
        })


# ========================================================
# 8. Zapis wyników
# ========================================================

df_lstm = pd.DataFrame(results)

os.makedirs("../results", exist_ok=True)

df_lstm.to_csv("../results/lstm_ae_window5_results.csv", index=False)

print("\n=== WYNIKI LSTM AE ===")
print(df_lstm[[
    "scenario",
    "LSTM_detected",
    "LSTM_total_windows",
    "LSTM AE DR (%)"
]])

print("\nZapisano: ../results/lstm_ae_window5_results.csv")
print("Zapisano model: ../models/lstm_ae_window5_best.pth")